In [1]:
import re
import pandas as pd
from pathlib import Path

TRANSCRIPT_DIR = Path("all_csv_nlp")

In [2]:
train = pd.read_csv("train_split_Depression_AVEC2017.csv")
dev = pd.read_csv("dev_split_Depression_AVEC2017.csv")
test = pd.read_csv("full_test_split.csv").dropna(how="all")

# the test file uses different column names
test = test.rename(columns={"PHQ_Binary": "PHQ8_Binary", "PHQ_Score": "PHQ8_Score"})

labels = pd.concat(
    [d[["Participant_ID", "PHQ8_Binary"]].assign(split=s)
     for d, s in [(train, "train"), (dev, "dev"), (test, "test")]],
    ignore_index=True,
).rename(columns={"Participant_ID": "id", "PHQ8_Binary": "label"})

labels["id"] = labels["id"].astype(int)
labels["label"] = labels["label"].astype(int)
print(labels.groupby(["split", "label"]).size().unstack())

label   0   1
split        
dev    23  12
test   33  14
train  77  30


In [3]:
def load_text(path):
    with open(path, encoding="utf-8") as f:
        header = f.readline()
    if "\t" in header:                       # standart DAIC-WOZ formatı
        t = pd.read_csv(path, sep="\t")
        lines = t.loc[t["speaker"] == "Participant", "value"]
    else:                                    # 440: virgülle ayrılmış, sadece participant konuşması
        t = pd.read_csv(path)
        lines = t["Text"]
    text = " ".join(lines.dropna().astype(str))
    text = re.sub(r"<[^>]*>", " ", text).lower()
    return re.sub(r"\s+", " ", text).strip()

texts = {int(p.name.split("_")[0]): load_text(p)
         for p in TRANSCRIPT_DIR.glob("*_TRANSCRIPT.csv")}
print(len(texts), "transcripts loaded")

189 transcripts loaded


In [4]:
df = labels.assign(text=labels["id"].map(texts))

missing = df[df["text"].isna()]["id"].tolist()
print("missing transcripts:", missing)

df = df.dropna(subset=["text"]).reset_index(drop=True)
df["n_words"] = df["text"].str.split().str.len()

print(df.groupby(["split", "label"]).size().unstack())
print(df["n_words"].describe().round(1))
df.head()

missing transcripts: []
label   0   1
split        
dev    23  12
test   33  14
train  77  30
count     189.0
mean     1445.2
std       801.5
min       165.0
25%       874.0
50%      1249.0
75%      1798.0
max      4551.0
Name: n_words, dtype: float64


,id,label,split,text,n_words
0,303,0,train,okay how 'bout yourself here in california yea...,1960
1,304,0,train,i'm doing good um from los angeles california ...,976
2,305,0,train,i'm doing alright uh originally i'm from calif...,3321
3,310,0,train,yes it's okay fine i live in hollywood uh seat...,1069
4,312,0,train,yes fine how about you here yes the weather we...,1192


In [5]:
import os
os.makedirs("data", exist_ok=True)
df.to_csv("data/daicwoz.csv", index=False)
print(df.shape)
print(df.groupby(["split", "label"]).size().unstack())

(189, 5)
label   0   1
split        
dev    23  12
test   33  14
train  77  30


In [6]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import RandomizedSearchCV, RepeatedStratifiedKFold, StratifiedKFold, cross_val_predict
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, f1_score, roc_auc_score

df = pd.read_csv("data/daicwoz.csv")
train_dev = df[df["split"].isin(["train", "dev"])]
test = df[df["split"] == "test"]

def evaluate(y_true, y_prob, thr=0.5):
    y_pred = (np.asarray(y_prob) >= thr).astype(int)
    p, r, f, _ = precision_recall_fscore_support(y_true, y_pred, average="binary", zero_division=0)
    return {"accuracy": accuracy_score(y_true, y_pred), "precision": p, "recall": r, "f1": f,
            "macro_f1": f1_score(y_true, y_pred, average="macro"), "auroc": roc_auc_score(y_true, y_prob)}

pipe = Pipeline([
    ("tfidf", TfidfVectorizer()),
    ("clf", LogisticRegression(solver="liblinear", class_weight="balanced",
                               max_iter=2000, random_state=42)),
])

shared = {
    "tfidf__min_df": [1, 2, 3],
    "tfidf__max_df": [0.8, 0.95, 1.0],
    "tfidf__sublinear_tf": [True, False],
    "tfidf__use_idf": [True, False],
    "clf__penalty": ["l1", "l2"],
    "clf__C": [0.01, 0.1, 1, 10, 100],
}
param_space = [
    {**shared, "tfidf__analyzer": ["word"],
     "tfidf__ngram_range": [(1, 1), (1, 2), (1, 3)],
     "tfidf__stop_words": [None, "english"],
     "tfidf__max_features": [None, 500, 2000]},
    {**shared, "tfidf__analyzer": ["char_wb"],
     "tfidf__ngram_range": [(2, 4), (3, 5)],
     "tfidf__max_features": [None, 5000]},
]

cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=42)
search = RandomizedSearchCV(pipe, param_space, n_iter=120, scoring="roc_auc",
                            cv=cv, random_state=42, n_jobs=-1)
search.fit(train_dev["text"], train_dev["label"])
print("Best CV AUROC:", round(search.best_score_, 3))
print("Best params:", search.best_params_)

best = search.best_estimator_
oof = cross_val_predict(best, train_dev["text"], train_dev["label"],
                        cv=StratifiedKFold(5, shuffle=True, random_state=42),
                        method="predict_proba")[:, 1]
thresholds = np.linspace(0.05, 0.95, 91)
scores = [f1_score(train_dev["label"], (oof >= t).astype(int), average="macro") for t in thresholds]
thr = thresholds[int(np.argmax(scores))]
print(f"Best threshold: {thr:.2f} (CV macro_f1={max(scores):.3f})")

best.fit(train_dev["text"], train_dev["label"])
pd.DataFrame({"test": evaluate(test["label"], best.predict_proba(test["text"])[:, 1], thr)}).T.round(3)

Best CV AUROC: 0.769
Best params: {'tfidf__use_idf': True, 'tfidf__sublinear_tf': True, 'tfidf__stop_words': None, 'tfidf__ngram_range': (1, 3), 'tfidf__min_df': 1, 'tfidf__max_features': None, 'tfidf__max_df': 0.8, 'tfidf__analyzer': 'word', 'clf__penalty': 'l2', 'clf__C': 10}
Best threshold: 0.36 (CV macro_f1=0.772)


,accuracy,precision,recall,f1,macro_f1,auroc
test,0.723,0.545,0.429,0.48,0.646,0.742
